In [0]:
jdbc_url = "jdbc:sqlserver://103.207.1.87:1433;databaseName=Travelon;encrypt=true;trustServerCertificate=true"
df = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "TravelonCore.Tourist") \
    .option("user", "codesages") \
    .option("password", "teamcodesages") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()

display(df)

In [0]:
df.write.format("parquet") \
    .mode("overwrite") \
    .save("dbfs:/Volumes/dev/bronze/my_volume/raw/tourist/")

In [0]:
staged_df = spark.read.format("parquet") \
    .load("dbfs:/Volumes/dev/bronze/my_volume/raw/tourist/")

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .save("dbfs:/Volumes/dev/bronze/my_volume/bronze/tourist/")

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("dev.bronze.tourist")

In [0]:
%sql
DESCRIBE TABLE dev.bronze.tourist;

In [0]:
%sql
CREATE OR REPLACE TABLE dev.silver.tourist AS
SELECT *
FROM (
    SELECT
        TouristId                          AS tourist_id,
        TRIM(Name)                         AS name,
        INITCAP(TRIM(Nationality))         AS nationality,
        Contact                            AS contact,
        LOWER(TRIM(Email))                 AS email,
        TRIM(Gender)                       AS gender,
        TRIM(KycType)                      AS kyc_type,
        EmergencyContact                   AS emergency_contact,
        TRIM(Address)                      AS address,
        AgencyId                           AS agency_id,
        TRIM(UserType)                     AS user_type,
        PrimaryDeviceId                    AS primary_device_id,
        KycLast4                           AS kyc_last4,

        CURRENT_TIMESTAMP()                AS processed_at,

        ROW_NUMBER() OVER (
            PARTITION BY TouristId
            ORDER BY TouristId DESC
        ) AS rn

    FROM dev.bronze.tourist
)
WHERE rn = 1;

In [0]:
%sql
SELECT COUNT(*) FROM dev.silver.tourist;

In [0]:
%sql
CREATE OR REPLACE TABLE dev.gold.tourist_by_gender AS
SELECT
    gender,
    COUNT(*) AS total
FROM dev.silver.tourist
GROUP BY gender;